In [1]:
# base checkout of workflow directory is here:
BASE='/home/ctbrown/scratch3/2025-workflow-core99/'

# parquet files from 'sourmash gather' against 3,216 metagenomes
BASE_OUTPUTS=BASE+'/outputs.core2'

THRESHOLD_BP=20_000
N_METAGENOMES=3216

In [2]:
import polars as pl
import glob
import matplotlib.pyplot as plt

In [3]:
extract_species_from_name = (pl.col("match_name")
                             .str.split(' ')
                             .list.slice(1, 2)
                             .list.join(' '))

## Load Very Many Results from manysearch and gather

In [4]:
filenames = glob.glob(BASE_OUTPUTS + '/*.parquet')

manysearch_df = (pl.scan_parquet(filenames)
          .with_columns(species=extract_species_from_name)
          .filter(pl.col("intersect_hashes") >= 20)
          .select(["species", "match_name", "containment", "query_name", "intersect_hashes", "scaled"])).collect()

In [5]:
filenames2 = '/group/ctbrowngrp2/amhorst/2025-pigparadigm/results/gatherxgtdb+bins.singleton.20k/*.csv'


gather_df = (pl.scan_csv(filenames2)
             .filter(pl.col("intersect_bp") >= 20_000)
             .with_columns(match_name=pl.col("name"))
             .with_columns(species=extract_species_from_name)
             .select(['species', 'match_name', 'intersect_bp', 'unique_intersect_bp', 'query_name'])).collect()

In [6]:
len(gather_df)


2451267

In [7]:
len(manysearch_df)

2451267

In [8]:
gather_df.columns

['species', 'match_name', 'intersect_bp', 'unique_intersect_bp', 'query_name']

In [9]:
gather_df.select(['species', 'query_name', 'intersect_bp', 'unique_intersect_bp'])

species,query_name,intersect_bp,unique_intersect_bp
str,str,i64,i64
"""s__Physcousia sp900766785""","""ERR1135178""",2355000,2355000
"""s__Sodaliphilus sp004557565""","""ERR1135178""",2164000,2164000
"""s__Prevotella sp004556065""","""ERR1135178""",1866000,1866000
"""s__W0P33-017 sp022768825""","""ERR1135178""",1717000,1717000
"""s__Parabacteroides sp022775005""","""ERR1135178""",1620000,1620000
…,…,…,…
"""s__Oscillibacter sp945876285""","""SRR8960986""",20000,20000
"""s__RGIG8748 sp934500705""","""SRR8960986""",20000,20000
"""s__W0P33-017 sp945871045""","""SRR8960986""",20000,20000


In [10]:
manysearch_df = manysearch_df.with_columns(intersect_bp=pl.col("scaled")*pl.col('intersect_hashes'))

In [11]:
manysearch_df.select(['species', 'query_name', 'intersect_bp'])

species,query_name,intersect_bp
str,str,i64
"""s__Prevotella sp900316565""","""SRR11723728""",1879000
"""s__UBA2810 sp900317945""","""SRR11723728""",1832000
"""s__Megasphaera elsdenii""","""SRR11723728""",1782000
"""s__Prevotella sp900319905""","""SRR11723728""",1663000
"""s__UBA629 sp900316625""","""SRR11723728""",1603000
…,…,…
"""s__Clostridium butyricum_A""","""SRR8960611""",20000
"""s__Alloenteromonas sp022781015""","""SRR8960611""",20000
"""s__Hominimerdicola aceti""","""SRR8960611""",20000


## Are manysearch and gather results equivalent? Yes.

In [12]:
m_df = manysearch_df.select(['species', 'query_name', 'intersect_bp']).sort(by=['species', 'query_name'])

In [13]:
g_df = gather_df.select(['species', 'query_name', 'intersect_bp']).sort(by=['species', 'query_name'])

In [14]:
m_df.equals(g_df)

True

## Are `intersect_bp` and `unique_intersect_bp` identical in the gather results? YES.

In [15]:
gather_df = gather_df.with_columns(check_eq=(pl.col("intersect_bp") == pl.col("unique_intersect_bp")))
gather_df['check_eq'].all()

True